# Pokemon Morphology Evaluation

Quantifies whether a LeJEPA encoder embeds **morphology** (same creature across
color variants nearby) vs. **palette** (different creatures with similar colors
nearby). Pokemon-specific: uses the `pokemon_11k` filename convention to recover
ground-truth identity (`morph_id`), shiny flag, game/gender/frame.

Headline metrics (run on the **held-out test split**):
- **same-`morph_id` recall@k** — should go UP with relational-preserving color aug.
- **shiny-twin median rank** — should go DOWN (shiny becomes a near neighbor).
- **color–embedding correlation** — the color-dominance signature, should go DOWN.
- **effective rank** — descriptive covariate.

Point `CHECKPOINT_PATH` at the run you want to evaluate, then run all cells. To
compare experiments, set `CHECKPOINT_PATH` to each run's `lejepa_last.pth` in turn
(e.g. `checkpoints/lejepa_morph_colorjitter/lejepa_last.pth`) and compare the
printed summary.

In [ ]:
CHECKPOINT_PATH = "../checkpoints/lejepa_morph_colorjitter/lejepa_last.pth"  # <-- edit
USE_EMA = True   # evaluate the EMA weights if present (recommended)
SPLIT = "test"   # 'test' (held-out, recommended) or 'train'
N_IMAGES = None  # None = all images in the split

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset

from jepajitfusion.config import DataConfig, EncoderConfig
from jepajitfusion.data.datasets import get_dataset
from jepajitfusion.data.transforms import eval_transform, reverse_transform
from jepajitfusion.encoder.vit import VisionTransformer
from jepajitfusion.utils import get_device
from jepajitfusion.eval.morphology import (
    parse_pokemon_filename, retrieval_recall_at_k, shiny_neighbor_ranks,
    color_descriptor, color_vs_embedding_correlation, effective_rank,
)

device = get_device()
print(device)

## Load checkpoint and build encoder

In [ ]:
ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
print(f"run_id={ckpt.get('run_id')}  epoch={ckpt.get('epoch')}")

enc_cfg = EncoderConfig(**ckpt["encoder_config"]) if "encoder_config" in ckpt else EncoderConfig()
ds_cfg = DataConfig(**ckpt["dataset_config"]) if "dataset_config" in ckpt else DataConfig()

encoder = VisionTransformer(
    img_size=ds_cfg.img_size, patch_size=enc_cfg.patch_size,
    in_channels=ds_cfg.num_channels, embed_dim=enc_cfg.embed_dim,
    depth=enc_cfg.depth, num_heads=enc_cfg.num_heads, mlp_ratio=enc_cfg.mlp_ratio,
).to(device)

state = ckpt["model_state_dict"]
if USE_EMA and ckpt.get("ema_state_dicts"):
    # EMA stores the same encoder params; use the first (highest-decay) EMA copy.
    state = ckpt["ema_state_dicts"][0]
    print("Using EMA weights")
encoder.load_state_dict(state)
encoder.eval();

## Load split, parse identities, encode

In [ ]:
train_ds, test_ds = get_dataset(
    ds_cfg.name, transform=eval_transform(ds_cfg.img_size),
    data_dir=ds_cfg.data_dir, test_size=ds_cfg.test_size,
)
dataset = test_ds if SPLIT == "test" else train_ds
if N_IMAGES is not None:
    dataset = Subset(dataset, range(min(N_IMAGES, len(dataset))))
    samples = [dataset.dataset.samples[i] for i in dataset.indices]
else:
    samples = dataset.samples

parsed = [parse_pokemon_filename(path) for path, _ in samples]
morph_ids = np.array([p["morph_id"] for p in parsed])
species_ids = np.array([p["species_id"] for p in parsed])
print(f"{len(parsed)} images | {len(set(morph_ids))} morph_ids | "
      f"{int(sum(p['shiny'] for p in parsed))} shiny")

loader = DataLoader(dataset, batch_size=128, shuffle=False, num_workers=4)
embs, imgs01 = [], []
with torch.no_grad():
    for images, _ in loader:
        embs.append(encoder(images.to(device)).cpu())
        imgs01.append((images + 1) / 2)  # [-1,1] -> [0,1] for color descriptor
embeddings = torch.cat(embs).numpy()
images01 = torch.cat(imgs01)
print("embeddings:", embeddings.shape)

## Headline metrics

In [ ]:
KS = (1, 5, 10)
rec_morph = retrieval_recall_at_k(embeddings, morph_ids, ks=KS)
rec_species = retrieval_recall_at_k(embeddings, species_ids, ks=KS)
shiny = shiny_neighbor_ranks(embeddings, parsed)
color_lab = color_descriptor(images01)
corr = color_vs_embedding_correlation(embeddings, color_lab)
r_eff = effective_rank(embeddings)

print(f"Checkpoint: {ckpt.get('run_id')} (epoch {ckpt.get('epoch')}), split={SPLIT}\n")
print("same-morph_id retrieval:")
for k in KS:
    print(f"  recall@{k:<2}={rec_morph[k]['recall']:.3f}  precision@{k:<2}={rec_morph[k]['precision']:.3f}")
print("same-species_id retrieval:")
for k in KS:
    print(f"  recall@{k:<2}={rec_species[k]['recall']:.3f}  precision@{k:<2}={rec_species[k]['precision']:.3f}")
print(f"\nshiny-twin: median_rank={shiny['median_rank']:.1f}  "
      f"recall@10={shiny['recall_at_10']:.3f}  (n={shiny['n_pairs']})")
print(f"color<->embedding corr: pearson={corr['pearson']:.3f}  spearman={corr['spearman']:.3f}")
print(f"effective rank: {r_eff:.1f} / {embeddings.shape[1]}")

**Reading the numbers.** Higher same-`morph_id` recall and *lower* shiny-twin
median rank both mean the encoder groups a creature's color variants together.
A *lower* color–embedding correlation means palette is no longer the dominant axis.
The baseline (`color_aug=default`) is expected to show high color correlation and
high shiny-twin rank; the `colorjitter`/`lab` runs should improve all three.

## Nearest-neighbor inspection

Green title = neighbor is the **same creature** as the query; red = different.
`*` marks shiny sprites.

In [ ]:
N_QUERIES, N_NEIGHBORS = 6, 6
rev = reverse_transform()

x = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-12)
sim = x @ x.T
np.fill_diagonal(sim, -np.inf)

rng = np.random.default_rng(0)
# Prefer queries that actually have same-creature siblings to make the test fair.
has_sib = [i for i in range(len(parsed))
           if (morph_ids == morph_ids[i]).sum() > 1]
queries = rng.choice(has_sib, size=min(N_QUERIES, len(has_sib)), replace=False)

fig, axes = plt.subplots(N_QUERIES, N_NEIGHBORS + 1, figsize=(2 * (N_NEIGHBORS + 1), 2 * N_QUERIES))
for r, q in enumerate(queries):
    def show(ax, idx, title, color):
        ax.imshow(rev((images01[idx] * 2 - 1)))
        ax.set_title(title, fontsize=8, color=color)
        ax.axis("off")
    tag = lambda i: parsed[i]["morph_id"] + ("*" if parsed[i]["shiny"] else "")
    show(axes[r, 0], q, f"Q: {tag(q)}", "black")
    for c, nb in enumerate(np.argsort(-sim[q])[:N_NEIGHBORS]):
        same = morph_ids[nb] == morph_ids[q]
        show(axes[r, c + 1], nb, tag(nb), "green" if same else "red")
plt.tight_layout(); plt.show()